# LIVESTOCK INTELLIGENCE: OLAP Analysis & Early Warning System
## Laporan Praktikum TPD - Kelompok 1 (3SI1)

**Tujuan**: Menganalisis data terintegrasi BPS, iSIKHNAS, dan PIHPS untuk:
1. Early Warning System (Supply Risk Index)
2. Tren Harga vs Wabah Penyakit
3. Supply vs Demand Gap Analysis
4. Peta Risiko Spasial
5. Ketergantungan & Dependensi Supply

---

## SETUP: Import Libraries & Database Connection

In [ ]:
# Data processing & analysis
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics
from scipy import stats
from scipy.stats import pearsonr

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All libraries imported successfully")

### Setup Database Connection

In [ ]:
# Load environment variables
load_dotenv()

# Database configuration
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'datawarehouse_db')
DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create SQLAlchemy engine
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute('SELECT 1')
    print(f"✓ Connected to PostgreSQL: {DB_NAME}@{DB_HOST}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("Make sure .env file is configured correctly")

---
## SECTION 1: Data Loading & OLAP Cube Setup

Tahap ini memuat data dari fact_supply_resilience dan dimensi terkait untuk operasi OLAP.

In [ ]:
# Load main fact table
query_fact = """SELECT * FROM fact_supply_resilience ORDER BY waktu_key, prov_key"""
fact_data = pd.read_sql(query_fact, engine)

print(f"✓ Loaded Fact Table: {fact_data.shape[0]} rows × {fact_data.shape[1]} columns")
print(f"\nFact Table Columns:")
print(fact_data.dtypes)
print(f"\nFirst 5 rows:")
fact_data.head()

In [ ]:
# Load dimension tables
dim_prov = pd.read_sql("SELECT * FROM dim_prov", engine)
dim_waktu = pd.read_sql("SELECT * FROM dim_waktu", engine)
dim_komoditas = pd.read_sql("SELECT * FROM dim_komoditas", engine)

print(f"✓ Loaded Dimensions:")
print(f"  - dim_prov: {dim_prov.shape[0]} provinces")
print(f"  - dim_waktu: {dim_waktu.shape[0]} months")
print(f"  - dim_komoditas: {dim_komoditas.shape[0]} commodities")

# Display sample data
print(f"\nSample - Provinsi:")
print(dim_prov.head())

print(f"\nSample - Komoditas:")
print(dim_komoditas)

In [ ]:
# Create unified OLAP cube by joining all dimensions
olap_cube = fact_data.copy()
olap_cube = olap_cube.merge(dim_prov, on='prov_key', how='left')
olap_cube = olap_cube.merge(dim_waktu, on='waktu_key', how='left')
olap_cube = olap_cube.merge(dim_komoditas, on='komoditas_key', how='left')

print(f"✓ Created OLAP Cube with dimensions: {olap_cube.shape}")
print(f"\nCube Summary Statistics:")
print(olap_cube[['supply_risk_index', 'avg_harga', 'sum_jumlah_sakit', 
                  'sum_vol_mutasi', 'sum_realisasi_karkas']].describe().round(2))

---
## SECTION 2: OLAP OPERATIONS - Slicing, Dicing, Roll-up, Drill-down

### 2.1 SLICING: Filter by Single Dimension

In [ ]:
# Slicing Example 1: Filter by Commodity (e.g., Sapi only)
sapi_data = olap_cube[olap_cube['nama_komoditas'] == 'Sapi'].copy()
print(f"Slicing by Sapi: {sapi_data.shape[0]} records")

# Slicing Example 2: Filter by Province (e.g., Jawa Timur)
jatim_data = olap_cube[olap_cube['nama_provinsi'] == 'JAWA TIMUR'].copy()
print(f"Slicing by JAWA TIMUR: {jatim_data.shape[0]} records")

# Slicing Example 3: Filter by Year
year_2024_data = olap_cube[olap_cube['tahun'] == 2024].copy()
print(f"Slicing by Year 2024: {year_2024_data.shape[0]} records")

### 2.2 DICING: Filter by Multiple Dimensions

In [ ]:
# Dicing: Filter Sapi at Jawa Timur for Q4 2024
diced_data = olap_cube[
    (olap_cube['nama_komoditas'] == 'Sapi') &
    (olap_cube['nama_provinsi'] == 'JAWA TIMUR') &
    (olap_cube['kuartal'] == 'Q4') &
    (olap_cube['tahun'] == 2024)
]

print(f"Diced Data (Sapi + JAWA TIMUR + Q4 2024): {diced_data.shape[0]} records")
print(f"\nRisk Index values:")
print(diced_data[['nama_provinsi', 'nama_komoditas', 'tahun', 'bulan', 'supply_risk_index']])

### 2.3 ROLL-UP: Aggregate to Higher Level (Provinsi level)

In [ ]:
# Roll-up: Aggregate by Province & Commodity (across all time)
rollup_prov_comm = olap_cube.groupby(['nama_provinsi', 'nama_komoditas']).agg({
    'supply_risk_index': 'mean',
    'avg_harga': 'mean',
    'sum_jumlah_sakit': 'sum',
    'sum_vol_mutasi': 'sum',
    'sum_realisasi_karkas': 'sum',
    'populasi_ternak': 'mean'
}).reset_index().sort_values('supply_risk_index', ascending=False)

print("Roll-up: Average Risk Index by Province & Commodity")
print(rollup_prov_comm.head(10).round(3))

In [ ]:
# Roll-up: Aggregate by Year (Temporal aggregation)
rollup_temporal = olap_cube.groupby('tahun').agg({
    'supply_risk_index': 'mean',
    'avg_harga': 'mean',
    'sum_jumlah_sakit': 'sum',
    'sum_vol_mutasi': 'sum',
    'jumlah_penduduk': 'sum'
}).reset_index()

print("\nRoll-up: Risk Index by Year")
print(rollup_temporal.round(3))

### 2.4 DRILL-DOWN: Disaggregate to Lower Level (Month level)

In [ ]:
# Drill-down: From Year → Month (for selected province)
selected_prov = 'JAWA TIMUR'
drilldown = olap_cube[
    (olap_cube['nama_provinsi'] == selected_prov) &
    (olap_cube['tahun'] == 2024)
].sort_values(['bulan', 'nama_komoditas'])[[
    'tahun', 'bulan', 'nama_komoditas', 'supply_risk_index', 'avg_harga', 'sum_jumlah_sakit'
]]

print(f"Drill-down: Monthly detail for {selected_prov} (2024)")
print(drilldown.round(3).to_string())

---
## SECTION 3: ANALYSIS 1 - Early Warning System (Supply Risk Index)

**Tujuan**: Mengidentifikasi provinsi dan waktu dengan risiko pasokan tertinggi untuk intervensi dini.

In [ ]:
# 3.1 Top 10 Provinsi dengan Risiko Tertinggi (agregasi all time)
risk_by_prov = olap_cube.groupby('nama_provinsi').agg({
    'supply_risk_index': ['mean', 'max', 'count']
}).round(4)
risk_by_prov.columns = ['Avg_Risk', 'Max_Risk', 'Observations']
risk_by_prov = risk_by_prov.sort_values('Avg_Risk', ascending=False).head(10)

print("TOP 10 PROVINSI RISIKO TERTINGGI (All Time):")
print(risk_by_prov)

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
risk_by_prov['Avg_Risk'].plot(kind='barh', ax=ax, color='#d62728')
ax.set_xlabel('Average Supply Risk Index (0 = Safe, 1 = Critical)', fontsize=12, fontweight='bold')
ax.set_ylabel('Provinsi', fontsize=12, fontweight='bold')
ax.set_title('TOP 10 PROVINSI DENGAN RISIKO PASOKAN TERTINGGI', fontsize=14, fontweight='bold')
ax.axvline(x=0.5, color='orange', linestyle='--', linewidth=2, label='Warning Threshold')
ax.axvline(x=0.7, color='red', linestyle='--', linewidth=2, label='Critical Threshold')
ax.legend()
plt.tight_layout()
plt.show()

print("\n✓ Risk assessment completed")

In [ ]:
# 3.2 Risk Timeline: Temporal trend untuk 5 provinsi risiko tertinggi
top_risk_provs = risk_by_prov.head(5).index.tolist()

risk_timeline = olap_cube[
    olap_cube['nama_provinsi'].isin(top_risk_provs) &
    (olap_cube['tahun'] >= 2023)
].groupby(['tahun', 'bulan', 'nama_provinsi'])['supply_risk_index'].mean().reset_index()

# Create temporal key for sorting
risk_timeline['time_key'] = risk_timeline['tahun'] * 100 + risk_timeline['bulan']
risk_timeline = risk_timeline.sort_values('time_key')

# Plotly interactive chart
fig = px.line(
    risk_timeline,
    x='time_key',
    y='supply_risk_index',
    color='nama_provinsi',
    title='TIMELINE RISIKO PASOKAN - Top 5 Provinsi Berisiko (2023-2025)',
    labels={'supply_risk_index': 'Risk Index', 'time_key': 'Periode (YYMMs)'},
    markers=True
)
fig.add_hline(y=0.5, line_dash='dash', line_color='orange', annotation_text='Warning Level')
fig.add_hline(y=0.7, line_dash='dash', line_color='red', annotation_text='Critical Level')
fig.show()

In [ ]:
# 3.3 Current Status (Latest Month): Alert System
latest_data = olap_cube[
    olap_cube['waktu_key'] == olap_cube['waktu_key'].max()
].groupby('nama_provinsi').agg({
    'supply_risk_index': 'mean',
    'avg_harga': 'mean',
    'sum_jumlah_sakit': 'sum',
    'sum_vol_mutasi': 'sum'
}).reset_index().sort_values('supply_risk_index', ascending=False)

# Add alert status
def get_alert_status(risk_index):
    if risk_index >= 0.7:
        return '🔴 CRITICAL'
    elif risk_index >= 0.5:
        return '🟠 WARNING'
    elif risk_index >= 0.3:
        return '🟡 CAUTION'
    else:
        return '🟢 SAFE'

latest_data['Status'] = latest_data['supply_risk_index'].apply(get_alert_status)

print("CURRENT ALERT STATUS (Latest Month):\n")
print(latest_data[['nama_provinsi', 'supply_risk_index', 'Status']].to_string(index=False))

---
## SECTION 4: ANALYSIS 2 - Tren Harga vs Wabah Penyakit (Correlation Analysis)

**Tujuan**: Membuktikan apakah volatilitas harga disebabkan oleh wabah atau faktor lain.

In [ ]:
# 4.1 Aggregate by Month untuk time-series analysis
ts_data = olap_cube.groupby(['tahun', 'bulan', 'nama_komoditas']).agg({
    'avg_harga': 'mean',
    'sum_jumlah_sakit': 'sum',
    'sum_jumlah_mati': 'sum',
    'supply_risk_index': 'mean'
}).reset_index()

ts_data['time_key'] = ts_data['tahun'] * 100 + ts_data['bulan']
ts_data = ts_data.sort_values('time_key')

# Calculate month-over-month changes
ts_data['harga_pct_change'] = ts_data.groupby('nama_komoditas')['avg_harga'].pct_change() * 100
ts_data['sakit_pct_change'] = ts_data.groupby('nama_komoditas')['sum_jumlah_sakit'].pct_change() * 100

print("Time Series Sample (last 6 months):")
print(ts_data.tail(6).round(2))

In [ ]:
# 4.2 Hitung Pearson Correlation antara Harga dan Kesehatan
correlation_results = {}

for commodity in ts_data['nama_komoditas'].unique():
    comm_data = ts_data[ts_data['nama_komoditas'] == commodity].copy()
    comm_data = comm_data.dropna(subset=['avg_harga', 'sum_jumlah_sakit'])
    
    if len(comm_data) > 2:
        # Pearson correlation
        corr, p_value = pearsonr(comm_data['avg_harga'], comm_data['sum_jumlah_sakit'])
        
        correlation_results[commodity] = {
            'Correlation': corr,
            'P-Value': p_value,
            'N': len(comm_data)
        }

corr_df = pd.DataFrame(correlation_results).T
print("PEARSON CORRELATION: Harga vs Jumlah Sakit\n")
print(corr_df.round(4))

# Interpretation
for commodity, row in corr_df.iterrows():
    r = row['Correlation']
    p = row['P-Value']
    
    if abs(r) >= 0.7:
        strength = "KUAT"
    elif abs(r) >= 0.5:
        strength = "SEDANG"
    else:
        strength = "LEMAH"
    
    sig = "signifikan" if p < 0.05 else "tidak signifikan"
    direction = "positif" if r > 0 else "negatif"
    
    print(f"\n{commodity}: Korelasi {strength} {direction} ({sig})")
    if abs(r) >= 0.7:
        print(f"  ⚠️  Kenaikan harga SANGAT terkait dengan wabah penyakit!")
    elif abs(r) >= 0.5:
        print(f"  ⚠️  Kenaikan harga cukup terkait dengan wabah penyakit.")
    else:
        print(f"  ℹ️  Kenaikan harga kemungkinan besar bukan dari wabah (faktor lain dominan).")

In [ ]:
# 4.3 Dual-axis Time Series Visualization
for commodity in ts_data['nama_komoditas'].unique():
    comm_ts = ts_data[ts_data['nama_komoditas'] == commodity].copy()
    
    fig = make_subplots(
        rows=1, cols=1,
        specs=[[{"secondary_y": True}]]
    )
    
    # Add price trace
    fig.add_trace(
        go.Scatter(
            x=comm_ts['time_key'],
            y=comm_ts['avg_harga'],
            name='Harga Rata-rata',
            line=dict(color='blue', width=3),
            mode='lines+markers'
        ),
        secondary_y=False
    )
    
    # Add disease trace
    fig.add_trace(
        go.Bar(
            x=comm_ts['time_key'],
            y=comm_ts['sum_jumlah_sakit'],
            name='Jumlah Sakit',
            marker=dict(color='red', opacity=0.6)
        ),
        secondary_y=True
    )
    
    fig.update_layout(
        title=f'TREN HARGA vs WABAH PENYAKIT - {commodity}',
        xaxis_title='Periode (YYMMs)',
        yaxis_title='Harga (Rp)',
        yaxis2_title='Jumlah Hewan Sakit',
        hovermode='x unified',
        height=500
    )
    
    fig.show()

In [ ]:
# 4.4 Scatter plot dengan regression line
for commodity in ts_data['nama_komoditas'].unique():
    comm_data = ts_data[ts_data['nama_komoditas'] == commodity].copy()
    comm_data = comm_data.dropna(subset=['avg_harga', 'sum_jumlah_sakit'])
    
    fig = px.scatter(
        comm_data,
        x='sum_jumlah_sakit',
        y='avg_harga',
        trendline='ols',
        title=f'SCATTER: Harga vs Wabah - {commodity}',
        labels={
            'sum_jumlah_sakit': 'Jumlah Hewan Sakit',
            'avg_harga': 'Harga Rata-rata (Rp)'
        },
        hover_data=['tahun', 'bulan']
    )
    
    fig.show()

---
## SECTION 5: ANALYSIS 3 - Supply vs Demand Gap Analysis

**Tujuan**: Mengukur ketimpangan antara permintaan dan ketersediaan untuk mendeteksi defisit pasokan.

In [ ]:
# 5.1 Level Logistik: Mutasi vs Permintaan (dalam satuan Ekor)
gap_logistik = olap_cube.groupby(['tahun', 'bulan', 'nama_komoditas']).agg({
    'sum_vol_mutasi': 'sum',
    'avg_permintaan_bulanan': 'mean'
}).reset_index()

gap_logistik['gap_ekor'] = gap_logistik['sum_vol_mutasi'] - gap_logistik['avg_permintaan_bulanan']
gap_logistik['gap_pct'] = (gap_logistik['gap_ekor'] / gap_logistik['avg_permintaan_bulanan'] * 100).round(2)
gap_logistik['status'] = gap_logistik['gap_ekor'].apply(
    lambda x: '✓ SURPLUS' if x > 0 else '✗ DEFICIT'
)

print("SUPPLY VS DEMAND ANALYSIS - LEVEL LOGISTIK (Ekor)\n")
print(gap_logistik.sort_values('tahun', ascending=False).head(12).to_string(index=False))

# Summary statistics
summary_logistik = gap_logistik.groupby('nama_komoditas').agg({
    'gap_ekor': ['mean', 'min', 'max'],
    'gap_pct': 'mean'
}).round(2)

print("\nRINGKASAN GAP LOGISTIK:\n")
print(summary_logistik)

In [ ]:
# 5.2 Level Konsumsi: Karkas vs Konsumsi (dalam satuan Kg)
gap_konsumsi = olap_cube.groupby(['tahun', 'bulan', 'nama_komoditas']).agg({
    'sum_realisasi_karkas': 'sum',
    'avg_konsumsi_bulanan': 'mean'
}).reset_index()

gap_konsumsi['gap_kg'] = gap_konsumsi['sum_realisasi_karkas'] - gap_konsumsi['avg_konsumsi_bulanan']
gap_konsumsi['gap_pct'] = (gap_konsumsi['gap_kg'] / gap_konsumsi['avg_konsumsi_bulanan'] * 100).round(2)
gap_konsumsi['status'] = gap_konsumsi['gap_kg'].apply(
    lambda x: '✓ SURPLUS' if x > 0 else '✗ DEFICIT'
)

print("SUPPLY VS DEMAND ANALYSIS - LEVEL KONSUMSI (Kg)\n")
print(gap_konsumsi.sort_values('tahun', ascending=False).head(12).to_string(index=False))

# Summary statistics
summary_konsumsi = gap_konsumsi.groupby('nama_komoditas').agg({
    'gap_kg': ['mean', 'min', 'max'],
    'gap_pct': 'mean'
}).round(2)

print("\nRINGKASAN GAP KONSUMSI:\n")
print(summary_konsumsi)

In [ ]:
# 5.3 Visualisasi Gap Timeline
gap_logistik['time_key'] = gap_logistik['tahun'] * 100 + gap_logistik['bulan']

fig = px.bar(
    gap_logistik.sort_values('time_key'),
    x='time_key',
    y='gap_ekor',
    color='nama_komoditas',
    barmode='group',
    title='TIMELINE SUPPLY-DEMAND GAP - Level Logistik',
    labels={'gap_ekor': 'Gap (Ekor)', 'time_key': 'Periode'},
    color_discrete_map={'Sapi': '#FF6B6B', 'Ayam': '#4ECDC4'}
)

fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.show()

In [ ]:
# 5.4 Deficit Analysis by Province
deficit_by_prov = olap_cube.groupby(['nama_provinsi', 'nama_komoditas']).agg({
    'sum_vol_mutasi': 'sum',
    'avg_permintaan_bulanan': 'sum'
}).reset_index()

deficit_by_prov['gap'] = deficit_by_prov['sum_vol_mutasi'] - deficit_by_prov['avg_permintaan_bulanan']
deficit_by_prov['gap_pct'] = (deficit_by_prov['gap'] / deficit_by_prov['avg_permintaan_bulanan'] * 100).round(1)

# Filter deficits only
deficit_only = deficit_by_prov[deficit_by_prov['gap'] < 0].sort_values('gap')

print("TOP 10 WILAYAH DENGAN DEFICIT TERBESAR (Semua Periode)\n")
print(deficit_only[['nama_provinsi', 'nama_komoditas', 'gap', 'gap_pct']].head(10).to_string(index=False))

---
## SECTION 6: ANALYSIS 4 - Peta Risiko Spasial

**Tujuan**: Mengidentifikasi wilayah berisiko tinggi berdasarkan densitas populasi ternak dan kesehatan.

In [ ]:
# 6.1 Create spatial risk matrix
spatial_risk = olap_cube.groupby('nama_provinsi').agg({
    'populasi_ternak': 'mean',
    'sum_jumlah_sakit': 'sum',
    'supply_risk_index': 'mean',
    'sum_vol_mutasi': 'sum'
}).reset_index()

# Calculate disease density (cases per 1000 animals)
spatial_risk['disease_density'] = (
    spatial_risk['sum_jumlah_sakit'] / (spatial_risk['populasi_ternak'] / 1000)
).round(2)

# Create risk zone classification
def classify_risk_zone(risk_idx, density):
    if risk_idx >= 0.5 and density > spatial_risk['disease_density'].quantile(0.75):
        return 'RED ZONE - Prioritas Utama'
    elif risk_idx >= 0.5 or density > spatial_risk['disease_density'].median():
        return 'ORANGE ZONE - Perhatian'
    else:
        return 'GREEN ZONE - Stabil'

spatial_risk['risk_zone'] = spatial_risk.apply(
    lambda row: classify_risk_zone(row['supply_risk_index'], row['disease_density']),
    axis=1
)

print("SPATIAL RISK MATRIX - Wilayah Berdasarkan Populasi Ternak & Kesehatan\n")
print(spatial_risk.sort_values('supply_risk_index', ascending=False)
      [['nama_provinsi', 'populasi_ternak', 'sum_jumlah_sakit', 'disease_density', 'risk_zone']]
      .round(2).to_string(index=False))

In [ ]:
# 6.2 Risk Zone Distribution
zone_counts = spatial_risk['risk_zone'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'RED ZONE - Prioritas Utama': '#d62728', 
          'ORANGE ZONE - Perhatian': '#ff7f0e',
          'GREEN ZONE - Stabil': '#2ca02c'}

zone_counts.plot(kind='bar', ax=ax, color=[colors[z] for z in zone_counts.index], width=0.7)
ax.set_xlabel('Risk Zone Classification', fontsize=12, fontweight='bold')
ax.set_ylabel('Jumlah Provinsi', fontsize=12, fontweight='bold')
ax.set_title('DISTRIBUSI WILAYAH MENURUT ZONA RISIKO', fontsize=14, fontweight='bold')
ax.set_xticklabels(zone_counts.index, rotation=45, ha='right')

for i, v in enumerate(zone_counts.values):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Bubble chart: Populasi vs Disease vs Risk Index
fig = px.scatter(
    spatial_risk,
    x='populasi_ternak',
    y='disease_density',
    size='supply_risk_index',
    color='risk_zone',
    hover_name='nama_provinsi',
    title='PETA RISIKO SPASIAL - Populasi vs Disease Density (bubble size = Risk Index)',
    labels={
        'populasi_ternak': 'Populasi Ternak (ekor)',
        'disease_density': 'Densitas Penyakit (kasus per 1000 ekor)'
    },
    color_discrete_map={
        'RED ZONE - Prioritas Utama': '#d62728',
        'ORANGE ZONE - Perhatian': '#ff7f0e',
        'GREEN ZONE - Stabil': '#2ca02c'
    },
    size_max=50
)

fig.show()

In [ ]:
# 6.4 Prioritas Intervens Vaksinasi (High Risk + High Population)
intervention_priority = spatial_risk[
    (spatial_risk['supply_risk_index'] >= spatial_risk['supply_risk_index'].quantile(0.5)) |
    (spatial_risk['disease_density'] >= spatial_risk['disease_density'].quantile(0.75))
].sort_values(['supply_risk_index', 'disease_density'], ascending=False)

print("\nPRIORITAS ALOKASI PROGRAM KESEHATAN TERNAK (Vaksinasi/Intervensi):\n")
for idx, row in intervention_priority.iterrows():
    print(f"{'🔴' if row['risk_zone'] == 'RED ZONE - Prioritas Utama' else '🟠'} {row['nama_provinsi']}")
    print(f"   Populasi: {row['populasi_ternak']:,.0f} ekor | Disease Density: {row['disease_density']:.1f} kasus/1000")
    print(f"   Risk Index: {row['supply_risk_index']:.3f} | Status: {row['risk_zone']}\n")

---
## SECTION 7: ANALYSIS 5 - Ketergantungan & Dependensi Supply

**Tujuan**: Mengukur proporsi dominasi wilayah produsen untuk mengidentifikasi 'provinsi kunci'.

In [ ]:
# 7.1 Supply Concentration: Proporsi mutasi per provinsi
supply_concentration = olap_cube.groupby('nama_provinsi').agg({
    'sum_vol_mutasi': 'sum',
    'populasi_ternak': 'mean',
    'supply_risk_index': 'mean'
}).reset_index()

# Calculate national totals
national_total = supply_concentration['sum_vol_mutasi'].sum()
supply_concentration['pct_national_supply'] = (
    supply_concentration['sum_vol_mutasi'] / national_total * 100
).round(2)

supply_concentration = supply_concentration.sort_values('sum_vol_mutasi', ascending=False)

print("KONSENTRASI PASOKAN - SUPPLY DEPENDENCY ANALYSIS\n")
print(supply_concentration[['nama_provinsi', 'sum_vol_mutasi', 'pct_national_supply', 'supply_risk_index']]
      .round(3).to_string(index=False))

# Cumulative percentage
supply_concentration['cumsum_pct'] = supply_concentration['pct_national_supply'].cumsum()

print(f"\n\nKEY PROVINCES (Top 5 Suppliers):")
print(f"=" * 70)

for idx, row in supply_concentration.head(5).iterrows():
    alert = "⚠️" if row['supply_risk_index'] >= 0.5 else "✓"
    print(f"{alert} {row['nama_provinsi']:20} | Supply: {row['pct_national_supply']:6.2f}% | Cumulative: {row['cumsum_pct']:6.2f}%")
    if row['supply_risk_index'] >= 0.5:
        print(f"   ⚠️  CRITICAL: Provinsi ini berisiko tinggi, perlu backup supply plan!")
    print()

In [ ]:
# 7.2 Visualize Supply Concentration (Pareto Chart)
fig = make_subplots(
    rows=1, cols=1,
    specs=[[{"secondary_y": True}]]
)

top_suppliers = supply_concentration.head(15)

# Bar chart for supply volume
fig.add_trace(
    go.Bar(
        x=top_suppliers['nama_provinsi'],
        y=top_suppliers['sum_vol_mutasi'],
        name='Supply Volume',
        marker=dict(color='steelblue'),
        showlegend=True
    ),
    secondary_y=False
)

# Line chart for cumulative percentage
fig.add_trace(
    go.Scatter(
        x=top_suppliers['nama_provinsi'],
        y=top_suppliers['cumsum_pct'],
        name='Cumulative %',
        line=dict(color='red', width=3),
        mode='lines+markers',
        yaxis='y2'
    ),
    secondary_y=True
)

fig.update_layout(
    title='PARETO CHART: Konsentrasi Pasokan Nasional',
    xaxis_title='Provinsi',
    yaxis_title='Supply Volume (ekor)',
    yaxis2_title='Cumulative %',
    hovermode='x unified',
    height=600
)

fig.show()

In [ ]:
# 7.3 Supply Vulnerability Analysis
# Provinces yang supply > 10% dari nasional AND risk index > 0.5
vulnerable_suppliers = supply_concentration[
    (supply_concentration['pct_national_supply'] >= 10) &
    (supply_concentration['supply_risk_index'] >= 0.5)
]

print("\n⚠️  CRITICAL SUPPLY VULNERABILITY - Provinsi Kunci Berisiko Tinggi\n")
print("=" * 80)

if len(vulnerable_suppliers) > 0:
    for idx, row in vulnerable_suppliers.iterrows():
        print(f"\n🔴 PROVINSI: {row['nama_provinsi']}")
        print(f"   Supply Kontribusi: {row['pct_national_supply']:.1f}% dari Nasional")
        print(f"   Risk Index: {row['supply_risk_index']:.3f} (TINGGI)")
        print(f"   Populasi Ternak: {row['populasi_ternak']:,.0f} ekor")
        print(f"\n   ⚠️  REKOMENDASI:")
        print(f"       1. Segera identifikasi penyebab risiko (wabah/logistik)")
        print(f"       2. Siapkan provinsi backup sebagai alternatif supply")
        print(f"       3. Increase monitoring frekuensi kesehatan ternak")
        print(f"       4. Koordinasi dengan dinas terkait untuk contingency planning")
else:
    print("✓ Tidak ada provinsi kunci yang berisiko tinggi saat ini.")
    print("Status supply chain stabil.")

---
## SECTION 8: EXECUTIVE SUMMARY & INSIGHTS

In [ ]:
# Generate Executive Summary
print("\n" + "="*80)
print("EXECUTIVE SUMMARY - LIVESTOCK INTELLIGENCE OLAP ANALYSIS")
print("="*80)

# 1. Current Risk Status
print("\n1. CURRENT RISK STATUS")
print("-" * 80)
avg_risk = olap_cube['supply_risk_index'].mean()
critical_count = (olap_cube['supply_risk_index'] >= 0.7).sum()
warning_count = ((olap_cube['supply_risk_index'] >= 0.5) & (olap_cube['supply_risk_index'] < 0.7)).sum()

print(f"   Average Risk Index: {avg_risk:.3f} (0=Safe, 1=Critical)")
print(f"   Observations in CRITICAL zone (>0.7): {critical_count:,}")
print(f"   Observations in WARNING zone (0.5-0.7): {warning_count:,}")

# 2. Main Risk Drivers
print("\n2. MAIN RISK DRIVERS (Correlation with Risk Index)")
print("-" * 80)
risk_drivers = olap_cube[['avg_harga', 'sum_jumlah_sakit', 'sum_vol_mutasi', 'supply_risk_index']].corr()
print(risk_drivers['supply_risk_index'].sort_values(ascending=False)[:-1])

# 3. Price Volatility Insights
print("\n3. PRICE VOLATILITY INSIGHTS")
print("-" * 80)
for commodity in ['Sapi', 'Ayam']:
    comm_price = olap_cube[olap_cube['nama_komoditas'] == commodity]['avg_harga']
    cv = (comm_price.std() / comm_price.mean() * 100)  # Coefficient of variation
    print(f"   {commodity}: Coefficient of Variation = {cv:.1f}% (Higher = More Volatile)")

# 4. Supply-Demand Balance
print("\n4. SUPPLY-DEMAND BALANCE")
print("-" * 80)
for commodity in ['Sapi', 'Ayam']:
    comm_gap = gap_logistik[gap_logistik['nama_komoditas'] == commodity]['gap_ekor']
    surplus_months = (comm_gap > 0).sum()
    deficit_months = (comm_gap < 0).sum()
    print(f"   {commodity}:")
    print(f"      - Surplus months: {surplus_months}")
    print(f"      - Deficit months: {deficit_months}")
    print(f"      - Avg gap: {comm_gap.mean():,.0f} ekor (positive=surplus, negative=deficit)")

# 5. Key Recommendations
print("\n5. KEY RECOMMENDATIONS")
print("-" * 80)
top_risk_prov = olap_cube.groupby('nama_provinsi')['supply_risk_index'].mean().nlargest(3)
print(f"   a) FOCUS MONITORING on top 3 risk provinces:")
for prov, risk in top_risk_prov.items():
    print(f"      - {prov}: Risk Index {risk:.3f}")

print(f"\n   b) STRENGTHEN surveillance on disease outbreaks (Health Impact strongly correlated with supply risk)")
print(f"\n   c) ESTABLISH backup supply sources for provinces with >10% national supply contribution")
print(f"\n   d) IMPLEMENT dynamic price monitoring - anomalies may signal upstream supply stress")
print(f"\n   e) REVIEW logistics efficiency (Mutasi vs Permintaan) to identify bottlenecks")

print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)

---

## Notes & Metadata

- **Data Source**: PostgreSQL Data Warehouse (fact_supply_resilience + dimensions)
- **Analysis Period**: Dynamic (based on dim_waktu range)
- **Granularity**: Monthly level
- **Key Metrics**:
  - supply_risk_index: Integrated risk indicator (0-1 scale)
  - avg_harga: Market price from PIHPS
  - sum_jumlah_sakit: Disease reports from iSIKHNAS
  - sum_vol_mutasi: Livestock movement volume
  - sum_realisasi_karkas: Slaughter realization

- **Statistical Methods**: Pearson correlation, Min-Max scaling, time-series aggregation
- **Visualizations**: Interactive Plotly charts + static Matplotlib plots

---

**Report Generated**: Livestock Intelligence OLAP Analysis v1.0  
**By**: Kelompok 1 (3SI1) - STIS  
**Status**: ✓ Ready for Policy Decision Support